<a href="https://colab.research.google.com/github/MZiaAfzal71/Melting-Point-Prediction-of-Boronic-Acids/blob/main/Data%20Files/Scripts%20and%20Models/Comparative_Modeling_of_Molecular_Descriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Comparative Modeling of Molecular Descriptors for Boronic Acids

This notebook automates the training and evaluation of multiple regression models across a variety of molecular descriptor types for boronic acids.
It integrates several descriptor sources and machine learning algorithms into a single, clean workflow for easy comparison and reproducibility.

## 🧩 Descriptors Used

The following molecular representations are analyzed:

* ⚛️ Coulomb Matrix – captures atomic charge–distance relationships

* 🧬 Mordred 3D Descriptors – a large set of calculated 2D/3D chemical features

* 🧫 Morgan Fingerprint (ECFP4) – circular fingerprints encoding atom environments

* 💠 MACCS Keys – predefined structural fragment fingerprints

* 🔷 Our Descriptor – graph-based descriptor focused on atomic connectivity around boron

## ⚙️ Models Evaluated

Each descriptor dataset is tested with five regression models:

* 🧠 Support Vector Regressor (SVR)

* 🌳 Decision Tree Regressor (DT)

* 🌲 Random Forest Regressor (RF)

* 💡 LightGBM (LGBM)

* 🚀 XGBoost (XGB)

## 📊 Workflow Overview

1.   Load Descriptor Files from the /Excel Files/ directory.
2.   Split Data into training and validation sets (80/20).
3.   Train Models on each descriptor representation.
4.   Evaluate Performance using:
     * Mean Absolute Error (MAE)
     * Coefficient of Determination (R²)
5.   Save Results to a single summary file
     → Excel Files/All_Model_Results.xlsx



## 🧠 Note

This notebook provides an end-to-end comparative analysis, highlighting which descriptors and models perform best for melting point prediction of boronic acids.
The models run automatically in a loop for clarity and reproducibility.

## 🧩 1. Clone Repository and Navigate to Working Directory

The following cell clones the GitHub repository “Melting-Point-Prediction-of-Boronic-Acids” and changes the current working directory to the folder containing the data files, scripts, and pre-trained models used in this project.

In [1]:
!git clone https://github.com/MZiaAfzal71/Melting-Point-Prediction-of-Boronic-Acids
%cd Melting-Point-Prediction-of-Boronic-Acids/Data\ Files/Scripts\ and\ Models

Cloning into 'Melting-Point-Prediction-of-Boronic-Acids'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 187 (delta 34), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 24.77 MiB | 18.36 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/Melting-Point-Prediction-of-Boronic-Acids/Data Files/Scripts and Models


## 🧠 2. Imports

In [2]:
# ==============================================
# 🧠 Import all necessary libraries
# ==============================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
import os

## 📂 3. Define descriptors and models

In [3]:
# ==============================================
# 📂 Define all descriptor files and models to test
# ==============================================

descriptor_files = {
    "CoulombMatrix": "Excel Files/CoulombMatrix_BoronicAcids_Desc.xlsx",
    "Mordred": "Excel Files/Boronic_Mordred_3DC.xlsx",
    "Morgan": "Excel Files/Boronic_Morgan_fingerprint.xlsx",
    "MACCS": "Excel Files/Boronic_MACCS_fingerprint.xlsx",
    "Descriptor": "Excel Files/Boronic_Bonds_Desc_Boron_En.xlsx"
}

# Define machine learning models
models = {
    "SVR": SVR(kernel='rbf', C=1, gamma=0.9),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42),
    "LightGBM": "LightGBM",  # special handling below
    "XGBoost": XGBRegressor(random_state=42)
}

## ⚙️ 4. Define a helper function to train and evaluate

In [4]:
# ==============================================
# ⚙️ Helper function: Train and evaluate models
# ==============================================
def evaluate_model(model_name, model, X_train, X_valid, y_train, y_valid, X, y):
    """
    Train model, make predictions, and compute metrics.
    Handles LightGBM and SVR separately due to their specific needs.

    Parameters:
    - model_name (str): Name of the model (e.g., 'SVR', 'LightGBM')
    - model: Initialized model object
    - X_train, X_valid, y_train, y_valid: Split data
    - X, y: Full dataset (for computing overall metrics)

    Returns:
    - (mae, r2, y_pred): MAE, R², and predictions
    """

    # Special case for LightGBM
    if model_name == "LightGBM":
        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid)
        params = {
            'objective': 'regression',
            'metric': 'mae',
            'boosting_type': 'gbdt',
            'random_state': 42,
            'verbose': -1
        }
        model = lgb.train(params, train_data, valid_sets=[valid_data])
        y_pred = model.predict(X)
        mae = mean_absolute_error(y, y_pred)
        r2 = r2_score(y, y_pred)

    # Special case for SVR (uses normalized targets)
    elif model_name == "SVR":
        maxy = y.max()
        y_scaled = y / maxy
        X_train, X_valid, y_train, y_valid = train_test_split(X, y_scaled, train_size=0.8, random_state=42)
        model.fit(X_train, y_train)
        y_pred_scaled = model.predict(X)
        y_pred = y_pred_scaled * maxy
        mae = mean_absolute_error(y, y_pred)
        r2 = r2_score(y_scaled, y_pred_scaled)

    # General case for other models
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X)
        mae = mean_absolute_error(y, y_pred)
        r2 = r2_score(y, y_pred)

    return mae, r2, y_pred



## 🚀 5. Main loop to train all models on all descriptors

In [5]:
# ==============================================
# 🚀 Run all models for all descriptor datasets
# ==============================================
results_summary = []

for desc_name, file_path in descriptor_files.items():
    print(f"\n📘 Processing Descriptor: {desc_name}")
    chem_file = pd.read_excel(file_path)
    chem_file.fillna(0, inplace=True)

    X = chem_file.iloc[:, 3:]
    y = chem_file['Melting Point']

    X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, random_state=42)

    for model_name, model in models.items():
        print(f"   🔹 Training {model_name} on {desc_name}...")
        mae, r2, y_pred = evaluate_model(model_name, model, X_train, X_valid, y_train, y_valid, X, y)

        results_summary.append({
            "Descriptor": desc_name,
            "Model": model_name,
            "MAE": mae,
            "R2": r2
        })

results_df = pd.DataFrame(results_summary)
results_df.sort_values(by="MAE", inplace=True)
results_df.to_excel("Excel Files/All_Model_Results.xlsx", index=False)

print("\n✅ All models completed! Results saved to 'Excel Files/All_Model_Results.xlsx'.")



📘 Processing Descriptor: CoulombMatrix
   🔹 Training SVR on CoulombMatrix...
   🔹 Training DecisionTree on CoulombMatrix...
   🔹 Training RandomForest on CoulombMatrix...
   🔹 Training LightGBM on CoulombMatrix...
   🔹 Training XGBoost on CoulombMatrix...

📘 Processing Descriptor: Mordred
   🔹 Training SVR on Mordred...
   🔹 Training DecisionTree on Mordred...
   🔹 Training RandomForest on Mordred...
   🔹 Training LightGBM on Mordred...
   🔹 Training XGBoost on Mordred...

📘 Processing Descriptor: Morgan
   🔹 Training SVR on Morgan...
   🔹 Training DecisionTree on Morgan...
   🔹 Training RandomForest on Morgan...
   🔹 Training LightGBM on Morgan...
   🔹 Training XGBoost on Morgan...

📘 Processing Descriptor: MACCS
   🔹 Training SVR on MACCS...
   🔹 Training DecisionTree on MACCS...
   🔹 Training RandomForest on MACCS...
   🔹 Training LightGBM on MACCS...
   🔹 Training XGBoost on MACCS...

📘 Processing Descriptor: Descriptor
   🔹 Training SVR on Descriptor...
   🔹 Training DecisionTree

## 📊 6. Display summarized results

In [6]:
# ==============================================
# 📊 Display summarized results
# ==============================================
display(results_df)


,Descriptor,Model,MAE,R2
9,Mordred,XGBoost,7.258487,0.896294
4,CoulombMatrix,XGBoost,9.434358,0.835491
6,Mordred,DecisionTree,9.878512,0.803368
8,Mordred,LightGBM,9.970905,0.891062
11,Morgan,DecisionTree,10.172452,0.813645
3,CoulombMatrix,LightGBM,11.133595,0.864026
1,CoulombMatrix,DecisionTree,11.336364,0.736306
24,Descriptor,XGBoost,11.417395,0.834302
21,Descriptor,DecisionTree,11.539807,0.769707
14,Morgan,XGBoost,13.838496,0.872916


## 💾 7. Save individual model predictions

In [7]:
# Optional detailed results saving
output_dir = "../Results/"
os.makedirs(output_dir, exist_ok=True)

for desc_name, file_path in descriptor_files.items():
    chem_file = pd.read_excel(file_path)
    chem_file.fillna(0, inplace=True)

    X = chem_file.iloc[:, 3:]
    y = chem_file['Melting Point']
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, random_state=42)

    for model_name, model in models.items():
        mae, r2, y_pred = evaluate_model(model_name, model, X_train, X_valid, y_train, y_valid, X, y)
        results = pd.DataFrame({
            'Name': chem_file['Name'],
            'Observed': y,
            'Predicted': y_pred,
            'Difference': abs(y - y_pred)
        })
        results.to_excel(f"{output_dir}/{desc_name}_{model_name}_Results.xlsx", index=False)


## ✅ Final Output

This single Colab notebook will:

* Automatically loop through all descriptor files.

* Train five models (SVR, DT, RF, LGBM, XGB).

* Compute and save MAE and R² scores.

* Store both summary and per-model Excel outputs cleanly.